In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import sign # type: ignore

In [ ]:
class McCullochPittsFunction(autograd.Function):
    @staticmethod
    def forward(ctx, w: tr.Tensor, b: float, x: tr.Tensor) -> tr.Tensor:
        z = w @ x + b
        s = sign.sign(z)
        return s

    @staticmethod
    def backward(ctx, grad_output):
        # No gradients, since the function is not differentiable.
        return None, None, None


class McCullochPitts(nn.Module):
    def __init__(self, w: tr.Tensor, b: float = 0.0):
        super().__init__()
        self.w = w.clone().detach()
        self.b = tr.tensor(b)

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        y = McCullochPittsFunction.apply(self.w, self.b, x)
        return y

    def predict(self, x: tr.Tensor) -> int:
        y = int(self.forward(x).item())
        return y

In [ ]:
#
# AND
#
# X Y T    X  Y  T
# 0 0 0   -1 -1 <0     1.0*(-1) + 1.0*(-1) = -2.0 => b=-0.5
# 0 1 0   -1  1 <0     1.0*(-1) + 1.0*( 1) =  0.0 => b=-0.5
# 1 0 0    1 -1 <0     1.0*( 1) + 1.0*(-1) =  0.0 => b=-0.5
# 1 1 1    1  1 >0     1.0*( 1) + 1.0*( 1) =  2.0 => b=-0.5

def test_McCullochPitts_and():
    X = tr.tensor([
        [-1.0, -1.0],
        [-1.0, +1.0],
        [+1.0, -1.0],
        [+1.0, +1.0],
    ])

    T = [-1, -1, -1, +1]

    w = tr.tensor([1.0, 1.0]).t()
    b = -0.5
    m = McCullochPitts(w, b)

    for x, t in zip(X, T):
        y = m.predict(x)
        assert y == t

#
# NAND
#
# X Y T    X  Y  T     
# 0 0 1   -1 -1 >0    -1.0*(-1) + -1.0*(-1) =  2.0 => b=+0.5
# 0 1 1   -1  1 >0    -1.0*(-1) + -1.0*( 1) =  0.0 => b=+0.5
# 1 0 1    1 -1 >0    -1.0*( 1) + -1.0*(-1) =  0.0 => b=+0.5
# 1 1 0    1  1 <0    -1.0*( 1) + -1.0*( 1) = -2.0 => b=+0.5

def test_McCullochPitts_nand():
    X = tr.tensor([
        [-1.0, -1.0],
        [-1.0, +1.0],
        [+1.0, -1.0],
        [+1.0, +1.0],
    ])

    T = [+1, +1, +1, -1]

    w = tr.tensor([-1.0, -1.0]).t()
    b = 0.5
    m = McCullochPitts(w, b)

    for x, t in zip(X, T):
        y = m.predict(x)
        assert y == t


#
# OR
#
# X Y T    X  Y  T     
# 0 0 0   -1 -1 <0     1.0*(-1) + 1.0*(-1) = -2.0 => b=+0.5
# 0 1 0   -1  1 >0     1.0*(-1) + 1.0*( 1) =  0.0 => b=+0.5
# 1 0 0    1 -1 >0     1.0*( 1) + 1.0*(-1) =  0.0 => b=+0.5
# 1 1 1    1  1 >0     1.0*( 1) + 1.0*( 1) =  2.0 => b=+0.5

def test_McCullochPitts_or():
    X = tr.tensor([
        [-1.0, -1.0],
        [-1.0, +1.0],
        [+1.0, -1.0],
        [+1.0, +1.0],
    ])

    T = [-1, +1, +1, +1]

    w = tr.tensor([1.0, 1.0]).t()
    b = 0.5
    m = McCullochPitts(w, b)

    for x, t in zip(X, T):
        y = m.predict(x)
        assert y == t

if __name__ == "__main__":
    test_McCullochPitts_and()
    test_McCullochPitts_nand()
    test_McCullochPitts_or()
